In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')

import warnings
warnings.filterwarnings('ignore')

--------------------------------------------------------------------------------------------

### 1. Load dataset and understand the shape

* Obervations: There are total 1460 houses(rows(samples)) and 81 columns(features)in training set. and in test set there are total 1459 rows and 80 columns.
* Mean (180k) is much higher than the median (163k),this means there are few houses having higher price which are increasing the average this is causing skewness in dataset, its called as positive skewness (right skewness).

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

print(f"Train: {train.shape[0]} houses(rows), {train.shape[1]} columns")
print(f"Test:  {test.shape[0]} houses(rows), {test.shape[1]} columns")
print(f"\nTarget — SalePrice:")
print(f"  Min:    ${train['SalePrice'].min():,}")
print(f"  Max:    ${train['SalePrice'].max():,}")
print(f"  Mean:   ${train['SalePrice'].mean():,.0f}")
print(f"  Median: ${train['SalePrice'].median():,.0f}")
train.head()

--------------------------------------------------------------------------------------------

### 2. Data Exploratory Analysis

In [ ]:
# Missing values

missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing


* Missing values in column PoolQC means the house has no pool, and not missing data.
* So we will encode NaN as "No_Pool" instead of imputing with mean/mode.

In [ ]:
missing_values = train.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
percentage = (missing / len(train) * 100).round(1)

missing_values_df = pd.DataFrame({'count': missing_values, 'percent': percentage})
print(f"{len(missing_values_df)} columns have missing values\n")
print(missing_values_df.head(15))

In [ ]:
missing_values_df['percent'].head(20).plot(kind='bar', color='#7F87DD', figsize=(12,4))
plt.title('Top 20 columns by %')
plt.ylabel('Missing %')
plt.tight_layout()
plt.show()

* As Linear regression assumes target is normally distributed it required to do Log transform. 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train['SalePrice'], bins=60, color='#D4537E', edgecolor='white')
axes[0].set_title('Raw SalePrice — notice the long right tail')
axes[0].set_xlabel('Price ($)')

axes[1].hist(np.log1p(train['SalePrice']), bins=60, color='#1D9E75', edgecolor='white')
axes[1].set_title('Log(SalePrice) — roughly bell-shaped')
axes[1].set_xlabel('log(Price)')

plt.tight_layout()
plt.show()

skew_raw = train['SalePrice'].skew()
skew_log = np.log1p(train['SalePrice']).skew()
print(f"Skewness before log: {skew_raw:.3f}  (above 0.5 = skewed)")
print(f"Skewness after log:  {skew_log:.3f}  (closer to 0 = better for linear models)")

In [ ]:
numeric = train.select_dtypes(include=[np.number]).columns.tolist()
corr = train[numeric].corr()['SalePrice'].drop('SalePrice')
corr = corr.abs().sort_values(ascending=False)

top15 = corr.head(15)
colors = ['#7F77DD' if v >= 0.5 else '#B4B2A9' for v in top15]
top15.plot(kind='barh', color=colors, figsize=(10,5))
plt.axvline(0.5, color='#D4537E', linestyle='--', alpha=0.7, label='Strong correlation (>0.5)')
plt.title('Top 15 features correlated with SalePrice')
plt.legend()
plt.tight_layout()
plt.show()

print("Top 5:")
for feat, val in corr.head(5).items():
    print(f"  {feat:20} r = {val:.3f}")

In [ ]:
top4 = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, feat in zip(axes, top4):
    ax.scatter(train[feat], train['SalePrice'],
               alpha=0.3, color='#7F77DD', s=12)
    r = train[[feat,'SalePrice']].corr().iloc[0,1]
    ax.set_title(f'{feat}\nr = {r:.2f}')
    ax.set_xlabel(feat)
    ax.set_ylabel('SalePrice')

plt.tight_layout()
plt.show()

outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 200000)]
print(f"Outliers found: {len(outliers)} rows")
print(outliers[['GrLivArea', 'SalePrice', 'Neighborhood']])

In [ ]:
# Remove the 2 outliers we spotted
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 200000))]
print(f"Train rows after outlier removal: {len(train)}")

# Combine for consistent preprocessing (categoricals only — no target stats)
all_data = pd.concat([train.drop('SalePrice', axis=1), test], axis=0).reset_index(drop=True)

# "None" = feature doesn't exist (not missing, absent)
none_fill = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
             'MasVnrType']
for col in none_fill:
    all_data[col] = all_data[col].fillna('None')

# 0 = no area / no count
zero_fill = ['GarageYrBlt','GarageArea','GarageCars',
             'BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF',
             'BsmtFullBath','BsmtHalfBath','MasVnrArea']
for col in zero_fill:
    all_data[col] = all_data[col].fillna(0)

# Remaining: median for numbers, mode for categories
for col in all_data.select_dtypes(include=[np.number]).columns:
    all_data[col] = all_data[col].fillna(all_data[col].median())
for col in all_data.select_dtypes(include=['object']).columns:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

print(f"Remaining nulls: {all_data.isnull().sum().sum()}")

In [ ]:
# TotalSF: what buyers actually care about is total livable area
all_data['TotalSF']     = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

# TotalBath: combine all bathroom counts into one signal
all_data['TotalBath']   = (all_data['FullBath'] + 0.5 * all_data['HalfBath'] +
                           all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath'])

# Age at time of sale — buyers pay for newer houses
all_data['HouseAge']    = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge']    = all_data['YrSold'] - all_data['YearRemodAdd']

# Was it ever remodelled? Binary signal
all_data['WasRemodeled'] = (all_data['YearBuilt'] != all_data['YearRemodAdd']).astype(int)

print("New features created:")
for f in ['TotalSF','TotalBath','HouseAge','RemodAge','WasRemodeled']:
    print(f"  {f}: mean={all_data[f].mean():.1f}, min={all_data[f].min():.0f}, max={all_data[f].max():.0f}")

In [ ]:
# Quality ratings have natural order: Poor < Fair < Typical < Good < Excellent
# Label-encode these (not one-hot) because the order matters
qual_map = {'None':0, 'Po':1, 'Fa':2, 'TA':3, 'Gd':4, 'Ex':5}
qual_cols = ['ExterQual','ExterCond','BsmtQual','BsmtCond',
             'HeatingQC','KitchenQual','FireplaceQu','GarageQual',
             'GarageCond','PoolQC']
for col in qual_cols:
    all_data[col] = all_data[col].map(qual_map).fillna(0)

# Everything else: one-hot encode (no natural order)
all_data = pd.get_dummies(all_data)

n_train = len(train)
X      = all_data[:n_train].values
X_test = all_data[n_train:].values
y      = np.log1p(train['SalePrice'].values)

print(f"Final feature count: {X.shape[1]}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Scale AFTER splitting — fit scaler on train only, apply to both
# This prevents the test set's statistics leaking into training
scaler = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)   # learns mean/std from train
X_val_s = scaler.transform(X_val)      # applies train's mean/std to val
X_test_s = scaler.transform(X_test)    # same for test

print(f"Train: {X_tr.shape[0]} samples")
print(f"Val:   {X_val.shape[0]} samples")
print(f"\nScaler fit on train only — val and test use TRAIN mean/std")
print(f"Train mean of feature 0 before scaling: {X_tr[:,0].mean():.2f}")
print(f"Train mean of feature 0 after  scaling: {X_tr_s[:,0].mean():.4f}  (≈ 0)")

In [ ]:
models = {
    'Linear Regression'  : (LinearRegression(), X_tr_s, X_val_s),
    'Ridge (α=10)'        : (Ridge(alpha=10), X_tr_s, X_val_s),
    'Random Forest'      : (RandomForestRegressor(n_estimators=100, random_state=42), X_tr, X_val),
}

results = {}
for name, (model, Xtr, Xv) in models.items():
    model.fit(Xtr, y_tr)
    val_pred = model.predict(Xv)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    cv = cross_val_score(model, Xtr, y_tr, cv=5,
                          scoring='neg_root_mean_squared_error')
    cv_rmse = -cv.mean()
    results[name] = {'val': val_rmse, 'cv': cv_rmse, 'model': model}
    print(f"{name:22} | Val RMSE: {val_rmse:.4f} | CV RMSE: {cv_rmse:.4f}")

In [ ]:
rf = results['Random Forest']['model']
feat_names = all_data.columns.tolist()
importances = pd.Series(rf.feature_importances_, index=feat_names)
top20 = importances.sort_values(ascending=False).head(20)

top20.plot(kind='barh', color='#1D9E75', figsize=(10,6))
plt.gca().invert_yaxis()
plt.title('Top 20 features by importance (Random Forest)')
plt.xlabel('Importance score')
plt.tight_layout()
plt.show()

print("Top 5 most important features:")
print(top20.head())

# Generate submission
preds = np.expm1(rf.predict(X_test))  # reverse log-transform
sub = pd.DataFrame({'Id': test['Id'], 'SalePrice': preds})
sub.to_csv('submission.csv', index=False)
print(f"\nSubmission saved. Price range: ${preds.min():,.0f} — ${preds.max():,.0f}")
print("Sanity check: does this range feel realistic for houses?")